
# Channel Estimation with Vision Transformer (ViT) + Defensive Distillation — Final (data.mat)

**Runs with real training epochs (Teacher=60, Student=40).**  
Uses the original `data.mat` (`trainData`, `trainLabels`, `valData`, `valLabels`).  
Includes normalization fix for different widths (480 vs 32), ViT (fixed residuals), defensive distillation, FGSM/PGD attacks, full plots, and a results summary.


In [1]:
# Imports
import os, numpy as np, matplotlib.pyplot as plt
from scipy.io import loadmat
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error
print('TensorFlow:', tf.__version__)

2025-11-08 23:47:19.366647: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-08 23:47:20.181900: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-08 23:47:20.186708: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-08 23:47:21.805521: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


TensorFlow: 2.13.1


In [2]:
# Load data.mat and reshape (N,H,1,W)->(N,H,W,1)
data = loadmat('data.mat')
assert 'trainData' in data and 'trainLabels' in data and 'valData' in data and 'valLabels' in data, 'data.mat must contain trainData/trainLabels/valData/valLabels'

X_train = np.array(data['trainData'])
Y_train = np.array(data['trainLabels'])
X_test  = np.array(data['valData'])
Y_test  = np.array(data['valLabels'])

def reshape_data(arr):
    # original: (N, H, 1, W) -> (N, H, W, 1)
    if arr.ndim == 4 and arr.shape[2] == 1:
        return np.transpose(arr, (0,1,3,2))
    return arr

X_train = reshape_data(X_train); Y_train = reshape_data(Y_train)
X_test  = reshape_data(X_test);  Y_test  = reshape_data(Y_test)

print('Shapes -> X_train', X_train.shape, 'Y_train', Y_train.shape, 'X_test', X_test.shape, 'Y_test', Y_test.shape)
INPUT_SHAPE = X_train.shape[1:]

# ==== FIXED NORMALIZATION ====
# compute mean/std over (batch, width) -> broadcast shape (1, H, 1, C), works for both W=480 and W=32
X_mean = X_train.mean(axis=(0,2), keepdims=True)
X_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-9
X_train = (X_train - X_mean) / X_std
X_test  = (X_test  - X_mean) / X_std

# Flatten Y for regression
Y_train_flat = Y_train.reshape((Y_train.shape[0], -1))
Y_test_flat  = Y_test.reshape((Y_test.shape[0], -1))
print('Flattened ->', Y_train_flat.shape, Y_test_flat.shape)

Shapes -> X_train (612, 14, 480, 1) Y_train (612, 14, 480, 1) X_test (612, 14, 32, 1) Y_test (612, 14, 32, 1)
Flattened -> (612, 6720) (612, 448)


In [3]:
# ViT definition (fixed residual shapes)
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)
    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        x = self.projection(patch)
        x = x + self.position_embedding(positions)
        return x

def create_vit_for_regression(input_shape, patch_size=2, projection_dim=64, transformer_layers=6, num_heads=4, mlp_head_units=[256,128], dropout=0.1, output_dim=None):
    H, W, C = input_shape
    assert H % patch_size == 0 and W % patch_size == 0, 'H and W must be divisible by patch_size'
    num_patches = (H // patch_size) * (W // patch_size)
    patch_dim = patch_size * patch_size * C

    inputs = layers.Input(shape=input_shape)
    patches = layers.Reshape((H//patch_size, patch_size, W//patch_size, patch_size, C))(inputs)
    patches = layers.Permute((1,3,2,4,5))(patches)
    patches = layers.Reshape((num_patches, patch_dim))(patches)

    x = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        x2 = layers.Add()([attn_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=[projection_dim*2, projection_dim], dropout_rate=dropout)  # ensure same dim for residual
        x = layers.Add()([x3, x2])

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Flatten()(x)
    for units in mlp_head_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout)(x)
    if output_dim is None:
        output_dim = Y_train_flat.shape[1]
    outputs = layers.Dense(output_dim, activation='linear')(x)
    return keras.Model(inputs, outputs, name='ViT_ChannelEstimator')

In [4]:
# Adversarial attacks
def fgsm_attack(model, x, y, epsilon=0.01):
    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        preds = model(x_tensor, training=False)
        loss = tf.reduce_mean(tf.keras.losses.mean_squared_error(y_tensor, preds))
    grad = tape.gradient(loss, x_tensor)
    adv_x = x_tensor + epsilon * tf.sign(grad)
    return tf.clip_by_value(adv_x, -5.0, 5.0).numpy()

def pgd_attack(model, x, y, epsilon=0.01, alpha=0.002, iters=10):
    x_adv = x.copy().astype('float32')
    x_orig = x.copy().astype('float32')
    for _ in range(iters):
        x_adv = fgsm_attack(model, x_adv, y, epsilon=alpha)
        diff = x_adv - x_orig
        diff = np.clip(diff, -epsilon, epsilon)
        x_adv = x_orig + diff
    return x_adv

In [ ]:
# Train Teacher
BATCH_SIZE = 64
EPOCHS_TEACHER = 10
LEARNING_RATE = 1e-3

teacher = create_vit_for_regression(INPUT_SHAPE, patch_size=2, projection_dim=64, transformer_layers=6, num_heads=4, mlp_head_units=[256,128], dropout=0.1, output_dim=Y_train_flat.shape[1])
teacher.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss='mse', metrics=['mae'])
teacher.summary()
hist_teacher = teacher.fit(X_train, Y_train_flat, validation_split=0.1, epochs=EPOCHS_TEACHER, batch_size=BATCH_SIZE)
teacher.save('vit_teacher_dataMAT_final.h5')

t_loss = hist_teacher.history.get('loss', [])
t_vloss = hist_teacher.history.get('val_loss', [])

Model: "ViT_ChannelEstimator"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 14, 480, 1)]         0         []                            
                                                                                                  
 reshape (Reshape)           (None, 7, 2, 240, 2, 1)      0         ['input_1[0][0]']             
                                                                                                  
 permute (Permute)           (None, 7, 240, 2, 2, 1)      0         ['reshape[0][0]']             
                                                                                                  
 reshape_1 (Reshape)         (None, 1680, 4)              0         ['permute[0][0]']             
                                                                               

2025-11-08 23:47:34.903060: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2890137600 exceeds 10% of free system memory.
2025-11-08 23:47:35.730863: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2890137600 exceeds 10% of free system memory.
2025-11-08 23:47:42.590261: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2890137600 exceeds 10% of free system memory.
2025-11-08 23:47:56.277064: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2890137600 exceeds 10% of free system memory.


In [ ]:
# Defensive Distillation: Train Student
TEMPERATURE = 5.0
ALPHA = 0.7
EPOCHS_STUDENT = 10

# Soft targets from teacher
t_preds = teacher.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
def soft_targets(preds, T):
    std = np.std(preds, axis=1, keepdims=True) + 1e-9
    logits = preds / std
    exps = np.exp(logits / T)
    return exps / np.sum(exps, axis=1, keepdims=True)

t_soft = soft_targets(t_preds, T=TEMPERATURE)

student = create_vit_for_regression(INPUT_SHAPE, patch_size=2, projection_dim=64, transformer_layers=6, num_heads=4, mlp_head_units=[256,128], dropout=0.1, output_dim=Y_train_flat.shape[1])

loss_hard = tf.keras.losses.MeanSquaredError()
loss_soft = tf.keras.losses.KLDivergence()
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train_flat, t_soft)).shuffle(1000).batch(BATCH_SIZE)
s_train, s_val = [], []

@tf.function
def train_step(xb, yb, sb):
    with tf.GradientTape() as tape:
        preds = student(xb, training=True)
        hard = loss_hard(yb, preds)
        preds_norm = preds / (tf.math.reduce_std(preds, axis=1, keepdims=True) + 1e-9)
        preds_soft = tf.nn.softmax(preds_norm / TEMPERATURE, axis=1)
        soft = loss_soft(sb, preds_soft)
        loss = (1.0-ALPHA)*hard + ALPHA*soft
    grads = tape.gradient(loss, student.trainable_variables)
    optimizer.apply_gradients(zip(grads, student.trainable_variables))
    return loss

for ep in range(EPOCHS_STUDENT):
    ls = []
    for xb, yb, sb in train_ds:
        ls.append(train_step(xb, yb, sb).numpy())
    s_train.append(float(np.mean(ls)))
    val_preds = student.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    s_val.append(float(mean_squared_error(Y_test_flat, val_preds)))
    if (ep+1) % 5 == 0 or ep == 0:
        print(f'Epoch {ep+1}/{EPOCHS_STUDENT} - train {s_train[-1]:.6f} - val {s_val[-1]:.6f}')

student.save('vit_student_dataMAT_final.h5')

In [ ]:
# Evaluation: clean and adversarial
pred_t_clean = teacher.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
pred_s_clean = student.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
mse_t_clean = mean_squared_error(Y_test_flat, pred_t_clean)
mse_s_clean = mean_squared_error(Y_test_flat, pred_s_clean)
print('Teacher MSE (clean):', mse_t_clean)
print('Student MSE (clean):', mse_s_clean)

X_adv = pgd_attack(teacher, X_test, Y_test_flat, epsilon=0.01, alpha=0.002, iters=10)
pred_t_adv = teacher.predict(X_adv, batch_size=BATCH_SIZE, verbose=0)
pred_s_adv = student.predict(X_adv, batch_size=BATCH_SIZE, verbose=0)
mse_t_adv = mean_squared_error(Y_test_flat, pred_t_adv)
mse_s_adv = mean_squared_error(Y_test_flat, pred_s_adv)
print('Teacher MSE (adv):', mse_t_adv)
print('Student MSE (adv):', mse_s_adv)

In [ ]:
# Plots: Curves and Bar
plt.figure(figsize=(6,4))
plt.plot(t_loss, label='Teacher Train'); plt.plot(t_vloss, label='Teacher Val')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('Teacher Loss'); plt.legend(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(s_train, label='Student Train (Combined)'); plt.plot(s_val, label='Student Val MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('Student Loss'); plt.legend(); plt.show()

labels = ['Teacher Clean','Student Clean','Teacher Adv','Student Adv']
vals = [mse_t_clean, mse_s_clean, mse_t_adv, mse_s_adv]
plt.figure(figsize=(6,4)); plt.bar(labels, vals); plt.title('MSE: Clean vs Adversarial'); plt.ylabel('MSE'); plt.xticks(rotation=15); plt.show()

In [ ]:
# Heatmaps: True / Teacher / Student / Adv (Real & Imag)
H, W, C = Y_test.shape[1:]
def vec2mat(v): return v.reshape(H, W, C)

nplot = min(3, X_test.shape[0])
for i in range(nplot):
    y_true = Y_test[i]
    t_pred = vec2mat(pred_t_clean[i])
    s_pred = vec2mat(pred_s_clean[i])
    t_advp = vec2mat(pred_t_adv[i])
    s_advp = vec2mat(pred_s_adv[i])

    fig, axs = plt.subplots(3, 4, figsize=(14, 9))
    fig.suptitle(f'Sample {i}')

    axs[0,0].imshow(y_true[...,0]); axs[0,0].set_title('True Real'); axs[0,0].axis('off')
    axs[0,1].imshow(y_true[...,1]); axs[0,1].set_title('True Imag'); axs[0,1].axis('off')

    axs[1,0].imshow(t_pred[...,0]); axs[1,0].set_title('Teacher Real'); axs[1,0].axis('off')
    axs[1,1].imshow(t_pred[...,1]); axs[1,1].set_title('Teacher Imag'); axs[1,1].axis('off')

    axs[2,0].imshow(s_pred[...,0]); axs[2,0].set_title('Student Real'); axs[2,0].axis('off')
    axs[2,1].imshow(s_pred[...,1]); axs[2,1].set_title('Student Imag'); axs[2,1].axis('off')

    axs[0,2].imshow(t_advp[...,0]); axs[0,2].set_title('Teacher Adv Real'); axs[0,2].axis('off')
    axs[0,3].imshow(t_advp[...,1]); axs[0,3].set_title('Teacher Adv Imag'); axs[0,3].axis('off')

    axs[1,2].imshow(s_advp[...,0]); axs[1,2].set_title('Student Adv Real'); axs[1,2].axis('off')
    axs[1,3].imshow(s_advp[...,1]); axs[1,3].set_title('Student Adv Imag'); axs[1,3].axis('off')

    axs[2,2].axis('off'); axs[2,3].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Summary: percentage improvements
def pct_improve(baseline, new):
    return 100.0 * (baseline - new) / max(baseline, 1e-12)

improve_clean = pct_improve(mse_t_clean, mse_s_clean)
improve_adv   = pct_improve(mse_t_adv, mse_s_adv)

print('\n=== SUMMARY ===')
print(f'Teacher MSE (clean):   {mse_t_clean:.6f}')
print(f'Student MSE (clean):   {mse_s_clean:.6f}  |  Improvement: {improve_clean:.2f}%')
print(f'Teacher MSE (adv):     {mse_t_adv:.6f}')
print(f'Student MSE (adv):     {mse_s_adv:.6f}  |  Improvement: {improve_adv:.2f}%')

# Small table-like print
rows = [
    ('Metric','Teacher','Student','Improvement %'),
    ('MSE Clean', f'{mse_t_clean:.6f}', f'{mse_s_clean:.6f}', f'{improve_clean:.2f}%'),
    ('MSE Adv',   f'{mse_t_adv:.6f}',   f'{mse_s_adv:.6f}',   f'{improve_adv:.2f}%'),
]
colw = [14, 14, 14, 16]
print('\n' + ' | '.join(str(r).ljust(w) for r,w in zip(rows[0], colw)))
print('-'*sum(colw))
for r in rows[1:]:
    print(' | '.join(str(c).ljust(w) for c,w in zip(r, colw)))